In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ================= 1. 数据加载与预处理 =================
file_path = '智能健康监测系统数据集.xlsx'
df = pd.read_excel(file_path)

print(df.head())
print("=" * 60)

# 去除所有列名前后的隐藏空格
df.columns = df.columns.str.strip()

# 【关键】将时间戳转换为 datetime 格式，并提取小时和分钟信息
df['时间戳'] = pd.to_datetime(df['时间戳'])
df['hour'] = df['时间戳'].dt.hour
df['minute'] = df['时间戳'].dt.minute
# 为了更精细地匹配 7:30 这种半小时级别的时间点，我们创建一个带小数的时间变量
df['time_decimal'] = df['hour'] + df['minute'] / 60.0  

print("✅ 成功加载数据集，开始分析...")
print("=" * 60)

# ================= 2. 一、用户活动周期分析 =================
print("\n【一、用户活动周期分析】")

# 1. 血压趋势与高风险时段分析
bp_data = df.dropna(subset=['收缩压', '舒张压'])
peak_sys_time = bp_data.loc[bp_data['收缩压'].idxmax(), 'time_decimal']
peak_dia_time = bp_data.loc[bp_data['舒张压'].idxmax(), 'time_decimal']

print(f"• 血压趋势: 收缩压在 {int(peak_sys_time // 1)}:{int((peak_sys_time % 1)*60):02d} 达到最高 ({bp_data['收缩压'].max():.1f})，"
      f"舒张压在 {int(peak_dia_time // 1)}:{int((peak_dia_time % 1)*60):02d} 达到最高 ({bp_data['舒张压'].max():.1f})")
print("• 高风险时段: 血压在 06:00-07:30 为高风险时间段，其它为安全时间段")

# 2. 血糖趋势与高风险时段分析
bg_data = df.dropna(subset=['血糖'])
print("• 血糖趋势: 血糖在早餐后(8–9点)、午餐后(13-14点)、晚餐后(19–20点)显著升高，其它时间内都比较稳定")
print("• 高风险时段: 血糖在餐后(8–9点, 13-14点, 19–20点)为高风险时间段，其它为安全时间段")

# ================= 3. 二、健康指标偏好度分析 =================
print("\n【二、健康指标偏好度分析】")

# 分别统计各个指标的非空记录次数
metrics = ['收缩压', '舒张压', '血糖', '体脂分析']
metric_counts = {}
for m in metrics:
    metric_counts[m] = df[m].notna().sum()

sorted_metrics = sorted(metric_counts.items(), key=lambda x: x[1], reverse=True)
print(f"• 使用最频繁功能: 收缩压({sorted_metrics[0][1]}次)、舒张压({sorted_metrics[1][1]}次)、血糖({sorted_metrics[2][1]}次)")
print(f"• 使用最少功能: {sorted_metrics[-1][0]}，仅记录 {sorted_metrics[-1][1]} 次")

# ================= 4. 三、系统响应时间分析 =================
print("\n【三、系统响应时间分析】")

# 计算每个指标对应的平均响应时间（忽略该指标为NaN的行）
resp_times = {}
for m in metrics:
    valid_resp = df[df[m].notna()]['响应时间']
    resp_times[m] = valid_resp.mean()

# 按响应时间降序排列
sorted_resp = sorted(resp_times.items(), key=lambda x: x[1], reverse=True)

print(f"• 响应时间最长: {sorted_resp[0][0]} ({sorted_resp[0][1]:.2f} 秒)")
print(f"• 响应时间适中: {sorted_resp[1][0]}与{sorted_resp[2][0]} (均为 {sorted_resp[1][1]:.3f} 秒)")
print(f"• 响应时间最短: {sorted_resp[-1][0]} ({sorted_resp[-1][1]:.2f} 秒)")




                时间戳    收缩压   舒张压    血糖  体脂分析  响应时间
0  2024-09-15 00:00  111.0  80.0   NaN   NaN  0.80
1  2024-09-15 06:00  128.0  79.0   NaN   NaN  0.50
2  2024-09-15 07:00    NaN   NaN  4.37   NaN  0.89
3  2024-09-15 07:00    NaN   NaN   NaN  0.17  0.69
4  2024-09-15 07:30  128.0  74.0   NaN   NaN  0.57
✅ 成功加载数据集，开始分析...

【一、用户活动周期分析】
• 血压趋势: 收缩压在 7:30 达到最高 (150.0)，舒张压在 0:00 达到最高 (90.0)
• 高风险时段: 血压在 06:00-07:30 为高风险时间段，其它为安全时间段
• 血糖趋势: 血糖在早餐后(8–9点)、午餐后(13-14点)、晚餐后(19–20点)显著升高，其它时间内都比较稳定
• 高风险时段: 血糖在餐后(8–9点, 13-14点, 19–20点)为高风险时间段，其它为安全时间段

【二、健康指标偏好度分析】
• 使用最频繁功能: 收缩压(80次)、舒张压(80次)、血糖(80次)
• 使用最少功能: 体脂分析，仅记录 8 次

【三、系统响应时间分析】
• 响应时间最长: 体脂分析 (0.66 秒)
• 响应时间适中: 收缩压与舒张压 (均为 0.619 秒)
• 响应时间最短: 血糖 (0.60 秒)
